## RAVEN ON CLASSIFICATION 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from scipy.sparse import issparse
from raven import raven


Starting Corrected RAVEN Pipeline [CSV]...
Loading CSV: datasets/df_destx.csv
Original shape: (167, 725)
Final Feature Shape: (167, 717)
Applying Log2(x+1) transformation...
Classes: ['fcd' 'hc']
Train samples: 133, Test samples: 34
Total features: 717

--- 🔵 BASELINE ---
Baseline Accuracy: 0.9706 (Using all 717 features)

--- 🟣 RAVEN (tau=0.1) ---
Building redundancy graph...
Identifying essential and redundant features...
Selected features: 17
Reduction: 97.6%

METRIC                    | ACCURACY  
----------------------------------------
Baseline (717 feats)      | 0.9706
RAVEN (17 feats)          | 1.0000


In [ ]:
# change source as intended.
SOURCE = "CSV"         
DATASET_ID = 1138       
CSV_PATH = "datasets/df_destx.csv" 
CSV_SEP = ','
RAVEN_SAMPLE = 33000
TAU_THRESHOLD = 0.1
TEST_SIZE = 0.2
RANDOM_STATE = 42


In [ ]:

print(f"Starting Corrected RAVEN Pipeline [{SOURCE}]...")
if SOURCE == "OPENML":
    print(f"Fetching OpenML dataset {DATASET_ID}...")
    dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser="auto")
    X_raw = dataset.data
    y_raw = dataset.target

    if issparse(X_raw):
        X_df = pd.DataFrame(X_raw.toarray())
    else:
        X_df = pd.DataFrame(X_raw)

    y_series = pd.Series(y_raw)

elif SOURCE == "CSV":
    print(f"Loading CSV: {CSV_PATH}")
    df = pd.read_csv(CSV_PATH, sep=CSV_SEP)

    # Detect label column automatically
    label_col = None
    for col in ["type", "class", "label", "phenotype", "target", "group"]:
        if col in df.columns:
            label_col = col
            break

    if label_col is None:
        raise ValueError("No label column found in CSV. Please check your column names.")

    y_series = df[label_col]
    X_df = df.drop(columns=[label_col])

else:
    raise ValueError("SOURCE must be 'OPENML' or 'CSV'")


In [ ]:
print(f"Original shape: {X_df.shape}")
X_numeric = X_df.apply(pd.to_numeric, errors="coerce")
X_numeric = X_numeric.select_dtypes(include=[np.number]).dropna(axis=1, how="all")

# Transpose if genes are rows (Crucial for 0.97 accuracy!)
if X_numeric.shape[0] > X_numeric.shape[1]:
    X_numeric = X_numeric.T
    print("Transposed data to Samples × Features")

print(f"Final Feature Shape: {X_numeric.shape}")


In [ ]:
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X_numeric)

# Log transform if data is non-negative
if np.min(X_imputed) >= 0:
    print("Applying Log2(x+1) transformation...")
    X_processed = np.log2(X_imputed + 1)
else:
    X_processed = X_imputed

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)


In [ ]:

y_series = y_series.astype(str)
le = LabelEncoder()
y_encoded = le.fit_transform(y_series)

print(f"Classes: {le.classes_}")


X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

# RAVEN needs a DataFrame
X_train_df = pd.DataFrame(X_train)

print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
print(f"Total features: {X_train.shape[1]}")


In [ ]:

print("\n BASELINE ")
rf_base = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_base.fit(X_train, y_train)

acc_base = accuracy_score(y_test, rf_base.predict(X_test))
print(f"Baseline Accuracy: {acc_base:.4f} (Using all {X_train.shape[1]} features)")


In [ ]:

print(f"\nRAVEN (tau={TAU_THRESHOLD}) ")
essential_indices, _ = raven(
    data=X_train_df,
    mode="df",
    sample_size=RAVEN_SAMPLE,
    tau=TAU_THRESHOLD
)

print(f"Selected features: {len(essential_indices)}")
print(f"Reduction: {100 - (len(essential_indices) / X_train.shape[1] * 100):.1f}%")


In [ ]:
# Protect against RAVEN returning 0 features
if len(essential_indices) == 0:
    print("RAVEN eliminated ALL features. Consider lowering TAU_THRESHOLD.")
else:
    X_train_raven = X_train[:, essential_indices]
    X_test_raven = X_test[:, essential_indices]

    rf_raven = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    rf_raven.fit(X_train_raven, y_train)

    acc_raven = accuracy_score(y_test, rf_raven.predict(X_test_raven))


In [ ]:

print(f"{'Baseline (' + str(X_train.shape[1]) + ' feats)':<25} | {acc_base:.4f}")
print(f"{'RAVEN (' + str(len(essential_indices)) + ' feats)':<25} | {acc_raven:.4f}")
